#Part A — Text Classification

In [1]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout
from tensorflow.keras.datasets import imdb
from tensorflow.keras.preprocessing.sequence import pad_sequences

VOCAB_SIZE = 10000
MAX_LEN    = 300
EMBED_DIM  = 64

(x_train, y_train), (x_test, y_test) = imdb.load_data(num_words=VOCAB_SIZE)
x_train = pad_sequences(x_train, maxlen=MAX_LEN, padding='post')
x_test  = pad_sequences(x_test,  maxlen=MAX_LEN, padding='post')

model = Sequential([
    Embedding(VOCAB_SIZE, EMBED_DIM, input_length=MAX_LEN),
    LSTM(128, return_sequences=True),
    Dropout(0.3),
    LSTM(64),
    Dropout(0.3),
    Dense(1, activation='sigmoid')
])
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
model.summary()

history = model.fit(x_train, y_train, epochs=5, batch_size=64,
                    validation_split=0.2, verbose=1)
loss, acc = model.evaluate(x_test, y_test)
print(f"Test Accuracy: {acc:.4f}")


17464789/17464789 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

Epoch 1/5
313/313 ━━━━━━━━━━━━━━━━━━━━ 375s 1s/step - accuracy: 0.5175 - loss: 0.6915 - val_accuracy: 0.5724 - val_loss: 0.6660
Epoch 2/5
313/313 ━━━━━━━━━━━━━━━━━━━━ 348s 1s/step - accuracy: 0.5687 - loss: 0.6487 - val_accuracy: 0.5696 - val_loss: 0.6460
Epoch 3/5
313/313 ━━━━━━━━━━━━━━━━━━━━ 349s 1s/step - accuracy: 0.5792 - loss: 0.6402 - val_accuracy: 0.5020 - val_loss: 0.6821
Epoch 4/5
313/313 ━━━━━━━━━━━━━━━━━━━━ 383s 1s/step - accuracy: 0.5929 - loss: 0.6292 - val_accuracy: 0.6160 - val_loss: 0.6277
Epoch 5/5
313/313 ━━━━━━━━━━━━━━━━━━━━ 371s 1s/step - accuracy: 0.5827 - loss: 0.6585 - val_accuracy: 0.5342 - val_loss: 0.6882
782/782 ━━━━━━━━━━━━━━━━━━━━ 163s 209ms/step - accuracy: 0.5362 - loss: 0.6866
Test Accuracy: 0.5362


#Part B — Sequence-to-Sequence (Addition)

In [4]:
import numpy as np
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, LSTM, Dense

# ── FIX 1: '\0' ko CHARS mein add kiya — START token ke liye ────
CHARS     = '\x00' + '0123456789+ '   # index 1=null, 2-12=digits/plus/space
char2idx  = {c: i for i, c in enumerate(CHARS)}
idx2char  = {i: c for c, i in char2idx.items()}
VOCAB     = len(CHARS)                # 13

# ── Data generation ──────────────────────────────────────────────
def generate_data(n=10000):
    questions, answers = [], []
    for _ in range(n):
        a, b  = np.random.randint(0, 100), np.random.randint(0, 100)
        q     = f"{a}+{b}".ljust(6)   # max "99+99 " → length 6
        ans   = str(a + b).ljust(3)   # max "198"    → length 3
        questions.append(q)
        answers.append(ans)
    return questions, answers

# ── FIX 2: encode safely — unknown char → index 0 (null) ────────
def encode(seq_list, maxlen):
    enc = np.zeros((len(seq_list), maxlen, VOCAB), dtype=np.float32)
    for i, seq in enumerate(seq_list):
        for t, c in enumerate(seq):
            idx = char2idx.get(c, 0)   # unknown char → 0 instead of KeyError
            enc[i, t, idx] = 1.0
    return enc

questions, answers = generate_data()

enc_q       = encode(questions, 6)                         # (N, 6,  13)

# FIX 3: '\0' is now index 0 — no KeyError
enc_ans_in  = encode(['\x00' + a for a in answers], 4)    # (N, 4,  13) decoder input
enc_ans_out = encode([a + '\x00' for a in answers], 4)    # (N, 4,  13) decoder target

print(f"enc_q shape      : {enc_q.shape}")
print(f"enc_ans_in shape : {enc_ans_in.shape}")
print(f"enc_ans_out shape: {enc_ans_out.shape}")

# ── Encoder ──────────────────────────────────────────────────────
enc_inputs         = Input(shape=(6, VOCAB), name="encoder_input")
_, state_h, state_c = LSTM(64, return_state=True, name="encoder_lstm")(enc_inputs)

# ── Decoder ──────────────────────────────────────────────────────
dec_inputs = Input(shape=(None, VOCAB), name="decoder_input")
dec_lstm   = LSTM(64, return_sequences=True, return_state=True, name="decoder_lstm")
dec_out, _, _ = dec_lstm(dec_inputs, initial_state=[state_h, state_c])
outputs    = Dense(VOCAB, activation='softmax', name="output_dense")(dec_out)

# ── Model ────────────────────────────────────────────────────────
model = Model([enc_inputs, dec_inputs], outputs)
model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)
model.summary()

# ── Train ────────────────────────────────────────────────────────
history = model.fit(
    [enc_q, enc_ans_in],
    enc_ans_out,
    batch_size=128,
    epochs=30,
    validation_split=0.1,
    verbose=1,
)
print("\nSeq2Seq training complete!")

# ── Inference — test karo ────────────────────────────────────────
def predict_addition(a, b):
    q   = f"{a}+{b}".ljust(6)
    inp = encode([q], 6)                          # (1, 6, 13)

    # Start token
    dec_in = encode(['\x00' + '  '], 4)          # (1, 4, 13)

    pred   = model.predict([inp, dec_in], verbose=0)
    result = ''.join(
        idx2char.get(np.argmax(pred[0, t]), '')
        for t in range(3)
    ).strip().replace('\x00', '')

    print(f"{a} + {b} = {result}  (expected: {a+b})")

print("\n── Inference Demo ──")
predict_addition(12, 34)
predict_addition(55, 47)
predict_addition(7,  93)
predict_addition(0,  0)
predict_addition(99, 99)

enc_q shape      : (10000, 6, 13)
enc_ans_in shape : (10000, 4, 13)
enc_ans_out shape: (10000, 4, 13)


Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ encoder_input       │ (None, 6, 13)     │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ decoder_input       │ (None, None, 13)  │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ encoder_lstm (LSTM) │ [(None, 64),      │     19,968 │ encoder_input[0]… │
│                     │ (None, 64),       │            │                   │
│                     │ (None, 64)]       │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ decoder_lstm (LSTM) │ [(None, None,     │     19,968 │ decoder_input[0]… │
│                     │ 64), (None, 64),  │            │ encoder_lstm[0][… │
│                     │ (None, 64)]       │            │ encoder_lstm[0][… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ output_dense        │ (None, None, 13)  │        845 │ decoder_lstm[0][… │
│ (Dense)             │                   │            │                   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 40,781 (159.30 KB)

 Trainable params: 40,781 (159.30 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/30
71/71 ━━━━━━━━━━━━━━━━━━━━ 6s 40ms/step - accuracy: 0.3765 - loss: 2.1024 - val_accuracy: 0.5343 - val_loss: 1.6511
Epoch 2/30
71/71 ━━━━━━━━━━━━━━━━━━━━ 1s 18ms/step - accuracy: 0.5423 - loss: 1.4418 - val_accuracy: 0.5497 - val_loss: 1.3190
Epoch 3/30
71/71 ━━━━━━━━━━━━━━━━━━━━ 1s 19ms/step - accuracy: 0.5519 - loss: 1.2775 - val_accuracy: 0.5518 - val_loss: 1.2423
Epoch 4/30
71/71 ━━━━━━━━━━━━━━━━━━━━ 1s 19ms/step - accuracy: 0.5561 - loss: 1.2270 - val_accuracy: 0.5518 - val_loss: 1.2073
Epoch 5/30
71/71 ━━━━━━━━━━━━━━━━━━━━ 1s 18ms/step - accuracy: 0.5595 - loss: 1.1939 - val_accuracy: 0.5590 - val_loss: 1.1772
Epoch 6/30
71/71 ━━━━━━━━━━━━━━━━━━━━ 1s 18ms/step - accuracy: 0.5654 - loss: 1.1686 - val_accuracy: 0.5575 - val_loss: 1.1531
Epoch 7/30
71/71 ━━━━━━━━━━━━━━━━━━━━ 1s 19ms/step - accuracy: 0.5730 - loss: 1.1361 - val_accuracy: 0.5727 - val_loss: 1.1260
Epoch 8/30
71/71 ━━━━━━━━━━━━━━━━━━━━ 2s 25ms/step - accuracy: 0.5854 - loss: 1.0898 - val_accuracy: 0.6037 - v